# WAS-Mamba on BraTS -- Training on Kaggle

Kaggle-specific version of `train_brats.ipynb`. Two things are different from Colab here:

1. **No download needed** -- BraTS2021 is already a public Kaggle dataset. Attach it to this notebook instead of downloading:
   Right sidebar -> **Add Input** -> search **"BRaTS 2021 Task 1"** -> add `dschettler8845/brats-2021-task1`.
   It mounts read-only at `/kaggle/input/brats-2021-task1/`.
2. **Checkpoints** go to `/kaggle/working/` -- this persists for the life of the interactive session, but is only kept permanently if you **Save Version** (top right) before the session ends. Kaggle GPU sessions also have a **weekly quota** (not just a per-session limit like Colab) -- check Settings for how much you have left.

**Before running:**
- Notebook Settings (right sidebar) -> **Accelerator** -> GPU T4 x2 (or P100)
- Notebook Settings -> **Internet** -> **On** (needed for `git clone` and `pip install`)
- Add the BraTS2021 dataset as described above

## 1. Get the code

In [ ]:
import os

REPO_URL = "github.com/sachinn854/Brain-Tumor-Segmentatiton.git"
REPO_DIR = "/kaggle/working/Brain-Tumor-Segmentatiton"

if not os.path.isdir(REPO_DIR):
    get_ipython().system(f'git clone https://{REPO_URL} {REPO_DIR}')
else:
    get_ipython().system(f'git -C {REPO_DIR} pull')

get_ipython().run_line_magic('cd', REPO_DIR)

## 2. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install dependencies

Same reasoning as the Colab notebooks: `--no-build-isolation` so the build sees Kaggle's pre-installed torch, and `TORCH_CUDA_ARCH_LIST` pinned to the actual GPU so `mamba-ssm`/`causal-conv1d` don't compile for ~10 architectures they'll never use.

In [ ]:
import torch

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
print(f"Building only for compute capability {major}.{minor}")

!pip install -q einops timm nibabel ninja packaging
!pip install -q causal-conv1d --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation

## 4. Point at the attached dataset and reorganize into the layout `BratsDataset` expects

`/kaggle/input/` is **read-only**, so cases have to be reorganized into `/kaggle/working/BraTS2021/<case_id>/<files>` (a writable copy) rather than in place. This scans for every `*_flair.nii.gz` / `*_seg.nii.gz` etc. under the attached input and copies them into the expected per-case-folder layout by the case-ID prefix in each filename -- works whether the attached dataset's internal layout is already per-case folders or flat.

In [ ]:
import glob
import re
import shutil

KAGGLE_INPUT = '/kaggle/input/brats-2021-task1'
DATA_ROOT = '/kaggle/working/BraTS2021'

if not os.path.isdir(KAGGLE_INPUT):
    raise SystemExit(
        f"{KAGGLE_INPUT} not found -- add the dataset first: "
        "right sidebar -> Add Input -> search 'BRaTS 2021 Task 1' -> "
        "add dschettler8845/brats-2021-task1"
    )

if not os.path.isdir(DATA_ROOT):
    os.makedirs(DATA_ROOT, exist_ok=True)
    all_files = glob.glob(f'{KAGGLE_INPUT}/**/*.nii.gz', recursive=True)
    print(f'Found {len(all_files)} .nii.gz files under {KAGGLE_INPUT}')

    copied, skipped = 0, 0
    for fpath in all_files:
        fname = os.path.basename(fpath)
        match = re.match(r'(BraTS2021_\d+)_', fname)
        if not match:
            skipped += 1
            continue
        case_id = match.group(1)
        case_dir = os.path.join(DATA_ROOT, case_id)
        os.makedirs(case_dir, exist_ok=True)
        dest = os.path.join(case_dir, fname)
        if not os.path.exists(dest):
            shutil.copy2(fpath, dest)   # copy, not move -- /kaggle/input is read-only anyway
            copied += 1

    print(f'Copied {copied} files, skipped {skipped} (unrecognized naming)')
else:
    print(f'{DATA_ROOT} already exists, skipping.')

n_cases = len([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f'{n_cases} cases ready in {DATA_ROOT}')

If the dataset's internal layout turns out to be one big archive (e.g. a `.tar`) rather than loose `.nii.gz` files, this cell's glob will find nothing -- check what's actually under `/kaggle/input/brats-2021-task1/` first with `!find /kaggle/input/brats-2021-task1 -maxdepth 3 | head -20`, and extract any archive found there into `/kaggle/working/` before re-running the cell above.

## 5. Smoke-test the training loop (5 epochs)

In [ ]:
CHECKPOINT_DIR = '/kaggle/working/wasmamba_checkpoints'

!python -m src.engine.train \
    --data_path {DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5

If that ran cleanly with no OOM and printed per-epoch train/val loss and per-class Dice: the loop works on Kaggle's GPU too. Remove the smoke-test checkpoint before a real run, same reason as on Colab -- otherwise the next run "resumes" from epoch 5 instead of starting fresh.

In [ ]:
smoke_test_ckpt = os.path.join(CHECKPOINT_DIR, 'latest.pth')
if os.path.exists(smoke_test_ckpt):
    os.remove(smoke_test_ckpt)
    print('Removed smoke-test checkpoint. Ready for a real run.')
else:
    print('No checkpoint found -- nothing to remove.')

## 6. Real training

Remove `--epochs 5` for the paper's real 1000-epoch run once the smoke test above looks right. Given Kaggle's weekly GPU quota (not just a per-session cap), this will very likely span multiple sessions -- **Save Version before your session ends** so `/kaggle/working/wasmamba_checkpoints/latest.pth` isn't lost, then start a new session from that saved version to resume (this script auto-resumes from `latest.pth` if it's present).

In [ ]:
!python -m src.engine.train \
    --data_path {DATA_ROOT} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --epochs 5